In [ ]:
import requests
import pandas as pd
import json
import copy

## Preparation
Exploration of the SGU API collections — kept as reference, not needed to run.

In [ ]:
"""
url = "https://api.sgu.se/oppnadata/mineralrattigheter/ogc/features/v1/collections"
response = requests.get(url)
print(response.status_code)
print(response.headers.get("Content-Type"))
if response.status_code == 200:
    print("Good to go!")
    data = response.json()
else:
    print("Problem!")
"""

"""
# --- Before doing anything, let's check the structure ---
# Transforms "data" into a str in JSON format
print(json.dumps(data, indent=2))
# note: no ID as INT but a str
"""

"""
# --- Confirming IDs and titles as per above ---
# :40 does that the ID column has always up to 40 characters
for coll in data["collections"]:
    print(f"{coll['id']:40}  {coll['title']}")
"""

## Fetch the chosen collection

In [ ]:
collection = "ut-metaller-industrimineral-beviljade"
url = f"https://api.sgu.se/oppnadata/mineralrattigheter/ogc/features/v1/collections/{collection}/items"

query = {
    "limit": 827,
    "f": "json"
}

response = requests.get(url, params=query, verify=False)

print(response.status_code)
if response.status_code == 200:
    print("OK!")
else:
    print("Problem!")

## Raw data + backup

In [ ]:
raw_data = response.json()
raw_data_backup = copy.deepcopy(raw_data)
print("Backup done. Keys:", raw_data.keys())

## Create DataFrame + clean columns

In [ ]:
df = pd.json_normalize(raw_data["features"])

removed_columns = [
    "type", "geometry_name", "geometry.type", "geometry.coordinates",
    "properties.diarynr", "properties.status", "properties.export_date",
    "properties.geom_area", "properties.geom_length"
]

df_col_ok = df.drop(columns=removed_columns)
print(f"Kept columns ({len(df_col_ok.columns)}): {list(df_col_ok.columns)}")
df_col_ok.head()

## Exploratory Data Analysis (EDA)

In [ ]:
# --- Missing values ---
print("----- Missing values -----")
df_col_ok.isna().sum()

In [ ]:
# --- Counties with the most approvals ---
print("----- Counties with the most approvals -----")
df_col_ok["properties.county"].value_counts().head(10)

In [ ]:
# --- Mineral count (exploded) ---
print("----- Minerals count -----")
mineral_count = (
    df_col_ok["properties.mineral"]
    .str.split(",")     # str becomes a list as ["gold", "silver", etc..]
    .explode()          # separates in different rows --> 1 mineral per row
    .str.strip()        # remove white spaces
    .value_counts()     # count each mineral
)
mineral_count.head(13)

## Date conversion

In [ ]:
date_cols = ["properties.appl_date", "properties.dec_date", "properties.validfrom", "properties.validto"]
df_col_ok[date_cols] = df_col_ok[date_cols].apply(pd.to_datetime)
df_col_ok[date_cols].dtypes

## Permitting length

In [ ]:
# --- Application to decision ---
df_col_ok["appl_to_dec"] = (df_col_ok["properties.dec_date"] - df_col_ok["properties.appl_date"]).dt.days / 365.25
print("----- Application → decision (years) -----")
df_col_ok["appl_to_dec"].describe().round(2)

In [ ]:
# --- Permit duration (validfrom to validto) ---
df_col_ok["permit_duration"] = (df_col_ok["properties.validto"] - df_col_ok["properties.validfrom"]).dt.days / 365.25
print("----- Permit duration (years) -----")
df_col_ok["permit_duration"].describe().round(2)

In [ ]:
# --- Avg permit duration by county ---
print("----- Avg permit duration by county (years) -----")
df_col_ok.groupby("properties.county")["permit_duration"].mean().round(2).sort_values(ascending=False)

## Counties × Minerals

In [ ]:
mineral_by_county = (
    df_col_ok[["properties.mineral", "properties.county"]]
    .assign(mineral=df_col_ok["properties.mineral"].str.split(", "))
    .explode("mineral")
    .assign(mineral=lambda x: x["mineral"].str.strip())
    .groupby(["properties.county", "mineral"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
mineral_by_county.head(20)